# India EV Charging Investment Opportunity Analysis

## Data Loading and Cleaning

This notebook prepares the final analysis dataset for identifying EV charging infrastructure investment opportunities across selected major EV-relevant Indian states/UTs.

The cleaning workflow combines three major inputs:

- Public EV charging station data
- State-wise economic indicators
- Year-wise EV registration and EV share data

The final output is a state-level master dataset containing EV demand, cumulative EV market size, charging infrastructure supply, charging gap metrics, charger quality indicators, and economic variables.


## Data Sources

| Dataset | Source | Purpose |
|---|---|---|
| EV Public Charging Stations | [Bureau of Energy Efficiency - E Mobility](https://beeindia.gov.in/WriteReadData/RTF1984/EV_PCS_Data_29277.pdf) | Charging station count, connector availability, charger type, and fast charger share |
| EV Registration Data | [India Climate and Energy Dashboard](https://iced.niti.gov.in/analytics/ice-and-ev-vehicle-registered/) | Annual EV registrations, cumulative EV registrations, EV share, and EV growth |
| State-wise GSDP and Per-Capita NSDP | [MoSPI State-wise SDP](https://www.mospi.gov.in/sites/default/files/press_releases_statements/Statewise_SDP.xls) | Economic strength indicators |
| Power Supply Position | [Central Electricity Authority - Power Supply](https://cea.nic.in/dashboard/?lang=en) | Explored as a power readiness indicator |

## 1. Project Setup

This section imports the required Python libraries and sets display options. The notebook uses Pandas and NumPy for data cleaning, along with Matplotlib, Seaborn, and Plotly for quick validation and visual checks.


In [1]:
# Import core libraries used for data cleaning, aggregation, and visualization checks
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import glob
import re
import os

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

## 2. Analysis Scope

This version focuses on 14 major EV-relevant Indian states/UTs. The selected regions include large EV markets, economically important states, and regions with different levels of public charging infrastructure maturity.

This is a focused Version 1 analysis, not a complete national ranking. Future work can expand the same framework to all Indian states and Union Territories.


In [2]:
# Focused Version 1 sample: major EV-relevant states/UTs selected for analysis
selected_states = [
    "Maharashtra",
    "Karnataka",
    "Uttar Pradesh",
    "Delhi",
    "Tamil Nadu",
    "Gujarat",
    "Rajasthan",
    "Kerala",
    "Telangana",
    "Madhya Pradesh",
    "Haryana",
    "West Bengal",
    "Andhra Pradesh",
    "Punjab"
]

## 3. Public Charging Station Data

The charging station dataset is used to measure public charging infrastructure availability. It includes information such as state, district, city, charger type, charger rating, ownership type, and number of connectors.


In [3]:
# Load public charging station infrastructure data

charger_df = pd.read_excel("../data/raw/EV_PCS_Data_29277.xlsx")



In [4]:
charger_df.head()

,CPO Name,Govt/Private,State,District,City/Village,Location,Latitude,Longitude,Types of Chargers Installed/ Connector,Charger Rating,Connector Rating,No. of Connector
0,IOCL,Govt.,Andaman & Nicobar,South Andamans,South Andaman,Iocl Dealer Govind Nagar Swaraj Dweep (Haveloc...,12.034186,92.992718,Type-II AC,7.4,7.4,1
1,IOCL,Govt.,Andaman & Nicobar,South Andamans,South Andaman,Iocl Dealer S.No.45/1 Bimblitan Port Blair,11.604053,92.694109,Type-II AC,7.4,7.4,1
2,IOCL,Govt.,Andaman & Nicobar,South Andamans,South Andaman,Marina Refueling Station Vill-Austinabad Po-Br...,11.637410,92.73516,LEV AC Charge point,3.3,3.3,1
3,Tata Power,Private,Andaman & Nicobar,South Andamans,Port Blair,"Gennext Motors , Pahargaon, Ground Floor New, ...",11.630305,92.724835,CCS-II,25.0,25,1
4,Ather,Private,Andhra Pradesh,Tirupati,Tirupati,"27, 962, Chittoor-Bangalore Rd, Near Tvs Showr...",13.208240,79.091059,LEV DC Charge Point (IS-17017-2-7),3.3,3.3,1


In [5]:
charger_df.columns

Index(['CPO Name', 'Govt/Private', 'State', 'District', 'City/Village',
       'Location', 'Latitude', 'Longitude',
       'Types of Chargers Installed/ Connector', 'Charger Rating',
       'Connector Rating', 'No. of Connector'],
      dtype='object')

In [6]:
charger_df.shape

(39640, 12)

In [7]:
charger_df['State'].unique()

array(['Andaman & Nicobar', 'Andhra Pradesh', 'Arunachal Pradesh',
       'ARUNACHAL PRADESH', 'Assam', 'ASSAM', 'Bihar', 'BIHAR',
       'Chandigarh', 'Chhattisgarh', 'Delhi', 'Goa', 'Gujarat', 'Haryana',
       'HARYANA', 'Himachal Pradesh', 'Jammu & Kashmir', 'Jharkhand',
       'Karnataka', 'Kerala', 'KERALA', 'Ladakh', 'Lakshadweep',
       'Madhya Pradesh', 'Maharashtra', 'Manipur', 'Meghalaya', 'Mizoram',
       'Nagaland', 'Odisha', 'Puducherry', 'Punjab', 'Rajasthan',
       'RAJASTHAN', 'Sikkim', 'Tamil Nadu', 'Tamil nadu', 'Telangana',
       'Tripura', 'UT OF D&NH AND D&D', 'UT of D&NH and D&D',
       'Uttar Pradesh', 'Uttar pradesh', 'Uttrakhand', 'West Bengal',
       'West bengal'], dtype=object)

### 3.1 Clean Charging Station State Names

State names are standardized before filtering and merging. This step reduces errors caused by inconsistent capitalization, extra spaces, and alternate state/UT naming formats.


In [8]:
# Standardize state names before filtering and merging
charger_df["State"] = (
    charger_df["State"]
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
    .str.title()
)

In [9]:
# Correct known state/UT naming inconsistencies from the source file
state_name_mapping = {
    "Tamil Nadu": "Tamil Nadu",
    "Tamil Nadu": "Tamil Nadu",
    "Uttar Pradesh": "Uttar Pradesh",
    "West Bengal": "West Bengal",

    "Uttrakhand": "Uttarakhand",
    "Ut Of D&Nh And D&D": "Dadra & Nagar Haveli and Daman & Diu",
    "Ut Of D&Nh And D&D": "Dadra & Nagar Haveli and Daman & Diu",
    "D&Nh And D&D": "Dadra & Nagar Haveli and Daman & Diu"
}

charger_df["State"] = charger_df["State"].replace(state_name_mapping)

In [10]:
sorted(charger_df["State"].dropna().unique())

['Andaman & Nicobar',
 'Andhra Pradesh',
 'Arunachal Pradesh',
 'Assam',
 'Bihar',
 'Chandigarh',
 'Chhattisgarh',
 'Dadra & Nagar Haveli and Daman & Diu',
 'Delhi',
 'Goa',
 'Gujarat',
 'Haryana',
 'Himachal Pradesh',
 'Jammu & Kashmir',
 'Jharkhand',
 'Karnataka',
 'Kerala',
 'Ladakh',
 'Lakshadweep',
 'Madhya Pradesh',
 'Maharashtra',
 'Manipur',
 'Meghalaya',
 'Mizoram',
 'Nagaland',
 'Odisha',
 'Puducherry',
 'Punjab',
 'Rajasthan',
 'Sikkim',
 'Tamil Nadu',
 'Telangana',
 'Tripura',
 'Uttar Pradesh',
 'Uttarakhand',
 'West Bengal']

In [11]:
missing_states = set(selected_states) - set(charger_df["State"].unique())
missing_states

set()

In [12]:
# Keep only the selected Version 1 states/UTs for the focused analysis
charger_df = charger_df[charger_df["State"].isin(selected_states)].copy()
charger_df = charger_df.reset_index(drop=True)

In [13]:
sorted(charger_df["State"].unique())

['Andhra Pradesh',
 'Delhi',
 'Gujarat',
 'Haryana',
 'Karnataka',
 'Kerala',
 'Madhya Pradesh',
 'Maharashtra',
 'Punjab',
 'Rajasthan',
 'Tamil Nadu',
 'Telangana',
 'Uttar Pradesh',
 'West Bengal']

In [14]:
charger_df["State"].nunique()

14

### 3.2 Standardize Charging Station Columns

The raw charging dataset contains long column names. They are renamed to shorter, consistent names that are easier to use in analysis and reporting.


In [15]:
# Rename columns to clean, analysis-friendly names
charger_df = charger_df.rename(columns={
    "CPO Name": "CPO_Name",
    "Govt/Private": "Ownership",
    "City/Village": "City",
    "Types of Chargers Installed/ Connector": "Charger_Type",
    "Charger Rating": "Charger_Rating",
    "Connector Rating": "Connector_Rating",
    "No. of Connector": "No_of_Connectors"
})

In [16]:
charger_df = charger_df.dropna(how="all")

In [17]:
charger_df = charger_df.dropna(subset=["State"])

In [18]:
charger_df = charger_df.reset_index(drop=True)

### 3.3 Create Charger-Level Features

Additional fields are created to identify fast chargers and approximate unique charging station locations. These features are later aggregated at the state/UT level.


In [19]:
# Classify chargers with rating >= 25 kW as fast chargers
charger_df["Fast_Charger"] = charger_df["Charger_Rating"] >= 25

In [20]:
# Create a station identifier to estimate unique charging station locations
charger_df["Station_ID"] = (
    charger_df["State"].astype(str).str.strip() + "_" +
    charger_df["District"].astype(str).str.strip() + "_" +
    charger_df["City"].astype(str).str.strip() + "_" +
    charger_df["Location"].astype(str).str.strip() + "_" +
    charger_df["Latitude"].astype(str) + "_" +
    charger_df["Longitude"].astype(str)
)

### 3.4 Aggregate Charging Infrastructure by State/UT

The charger-level data is aggregated into state-level metrics such as total charging stations, total connectors, fast charger count, average charger rating, district coverage, and ownership counts.


In [21]:
# Aggregate charger-level data into one row per state/UT
charger_state = (
    charger_df
    .groupby("State")
    .agg(
        Charging_Station_Count=("Station_ID", "nunique"),
        Total_Connectors=("No_of_Connectors", "sum"),
        Fast_Charger_Count=("Fast_Charger", "sum"),
        Avg_Charger_Rating=("Charger_Rating", "mean"),
        District_Count=("District", "nunique"),
        City_Count=("City", "nunique"),
        Private_Charger_Rows=("Ownership", lambda x: (x == "Private").sum()),
        Govt_Charger_Rows=("Ownership", lambda x: (x == "Govt.").sum())
    )
    .reset_index()
)

charger_state

,State,Charging_Station_Count,Total_Connectors,Fast_Charger_Count,Avg_Charger_Rating,District_Count,City_Count,Private_Charger_Rows,Govt_Charger_Rows
0,Andhra Pradesh,793,1089,340,23.229664,46,385,161,702
1,Delhi,1977,3678,395,11.808317,24,3,2784,691
2,Gujarat,1208,1792,574,26.449754,60,291,307,1116
3,Haryana,936,3055,807,15.968648,39,173,2013,864
4,Karnataka,6146,8512,761,8.630936,46,505,6288,1958
5,Kerala,1394,2262,487,18.126961,26,294,601,1439
6,Madhya Pradesh,1147,1432,335,17.510926,94,294,71,1128
7,Maharashtra,4183,6315,1240,15.794832,62,598,4109,1638
8,Punjab,715,931,226,19.780584,46,223,88,700
9,Rajasthan,1531,1928,513,17.890078,99,383,280,1393


In [22]:
charger_state.head(n=25)

,State,Charging_Station_Count,Total_Connectors,Fast_Charger_Count,Avg_Charger_Rating,District_Count,City_Count,Private_Charger_Rows,Govt_Charger_Rows
0,Andhra Pradesh,793,1089,340,23.229664,46,385,161,702
1,Delhi,1977,3678,395,11.808317,24,3,2784,691
2,Gujarat,1208,1792,574,26.449754,60,291,307,1116
3,Haryana,936,3055,807,15.968648,39,173,2013,864
4,Karnataka,6146,8512,761,8.630936,46,505,6288,1958
5,Kerala,1394,2262,487,18.126961,26,294,601,1439
6,Madhya Pradesh,1147,1432,335,17.510926,94,294,71,1128
7,Maharashtra,4183,6315,1240,15.794832,62,598,4109,1638
8,Punjab,715,931,226,19.780584,46,223,88,700
9,Rajasthan,1531,1928,513,17.890078,99,383,280,1393


### 3.5 Create Charging Infrastructure Indicators

Two derived indicators are created:

- `Connectors_per_Station`: average connector availability per charging station
- `Fast_Charger_Share`: share of charging connectors classified as fast chargers

These help capture both infrastructure quantity and charging quality.


In [23]:
# Create charging infrastructure quality and density indicators
charger_state["Connectors_per_Station"] = (
    charger_state["Total_Connectors"] / charger_state["Charging_Station_Count"]
)

charger_state["Fast_Charger_Share"] = (
    charger_state["Fast_Charger_Count"] / charger_state["Charging_Station_Count"]
) * 100

In [24]:
# Save cleaned charging infrastructure summary
charger_state.to_csv("../data/cleaned/charger_state_summary.csv", index=False)

In [25]:
master_df = charger_state
master_df.head()

,State,Charging_Station_Count,Total_Connectors,Fast_Charger_Count,Avg_Charger_Rating,District_Count,City_Count,Private_Charger_Rows,Govt_Charger_Rows,Connectors_per_Station,Fast_Charger_Share
0,Andhra Pradesh,793,1089,340,23.229664,46,385,161,702,1.373266,42.875158
1,Delhi,1977,3678,395,11.808317,24,3,2784,691,1.860395,19.979767
2,Gujarat,1208,1792,574,26.449754,60,291,307,1116,1.483444,47.516556
3,Haryana,936,3055,807,15.968648,39,173,2013,864,3.263889,86.217949
4,Karnataka,6146,8512,761,8.630936,46,505,6288,1958,1.384966,12.382037


In [26]:
master_df.shape

(14, 11)

In [27]:
master_df.isnull().sum()

State                     0
Charging_Station_Count    0
Total_Connectors          0
Fast_Charger_Count        0
Avg_Charger_Rating        0
District_Count            0
City_Count                0
Private_Charger_Rows      0
Govt_Charger_Rows         0
Connectors_per_Station    0
Fast_Charger_Share        0
dtype: int64

In [28]:
master_df.to_csv("../data/cleaned/master_ev_charger_merged.csv", index=False)

## 4. Economic Data

Economic indicators are added because investment attractiveness depends not only on EV adoption, but also on market size and purchasing power. This section extracts GSDP and per-capita NSDP indicators from the state-wise economy workbook.


In [29]:
# Inspect available sheets in the state economy workbook
xls = pd.ExcelFile("../data/raw/Statewise_SDP.xls")

xls.sheet_names

['SDP-Curr.', 'SDP-Con', 'NDP-Curr.', 'NDP-Con.', 'PC curr.', 'PC con.']

In [30]:
for sheet in xls.sheet_names:
    print("\nSheet:", sheet)
    display(pd.read_excel("../data/raw/Statewise_SDP.xls", sheet_name=sheet).head())


Sheet: SDP-Curr.


,GROSS STATE DOMESTIC PRODUCT AT CURRENT PRICES; BASE YEAR 2011-12,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25,Unnamed: 26,Unnamed: 27,Unnamed: 28
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,As on 17.03.2025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,GSDP - CURRENT PRICES (` in Crore),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,(% Growth over previous year),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,S. No.,State\UT,2011-12,2012-13,2013-14,2014-15,2015-16,2016-17,2017-18,2018-19,2019-20,2020-21,2021-22,2022-23,2023-24,2024-25,2012-13,2013-14,2014-15,2015-16,2016-17,2017-18,2018-19,2019-20,2020-21,2021-22,2022-23,2023-24,2024-25
4,(1),(2),(3),(4),(5),(6),(7),(8'),(9),('10),('11),('12),('13),('14),('15),('16),('17),('18),('19),('20),('21),('22),('23),('24),('25),('26),('27),('28),('29)



Sheet: SDP-Con


,GROSS STATE DOMESTIC PRODUCT AT CONSTANT (2011-12) PRICES; BASE YEAR 2011-12,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25,Unnamed: 26,Unnamed: 27,Unnamed: 28
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,As on 17.03.2025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,GSDP - CONSTANT PRICES (` in Crore),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,(% Growth over previous year),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,S. No.,State\UT,2011-12,2012-13,2013-14,2014-15,2015-16,2016-17,2017-18,2018-19,2019-20,2020-21,2021-22,2022-23,2023-24,2024-25,2012-13,2013-14,2014-15,2015-16,2016-17,2017-18,2018-19,2019-20,2020-21,2021-22,2022-23,2023-24,2024-25
4,(1),(2),(3),(4),(5),(6),(7),(8'),(9'),('10),('11),('12),('13),('14),('15),('16),('17),('18),('19),('20),('21),('22),('23),('24),('25),('26),('27),('28),('29)



Sheet: NDP-Curr.


...

,NET STATE DOMESTIC PRODUCT AT CONSTANT (2011-12) PRICES; BASE YEAR 2011-12,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25,Unnamed: 26,Unnamed: 27,Unnamed: 28
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,As on 17.03.2025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NSDP - CONSTANT PRICES (` in Crore),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,(% Growth over previous year),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,S. No.,State\UT,2011-12,2012-13,2013-14,2014-15,2015-16,2016-17,2017-18,2018-19,2019-20,2020-21,2021-22,2022-23,2023-24,2024-25,2012-13,2013-14,2014-15,2015-16,2016-17,2017-18,2018-19,2019-20,2020-21,2021-22,2022-23,2023-24,2024-25
4,(1),(2),(3),(4),(5),(6),(7),(8'),(9'),('10),('11),('12),('13),('14),('15),('16),('17),('18),('19),('20),('21),('22),('23),('24),('25),('26),('27),('28),('29)



Sheet: PC curr.


,PER CAPITA NET STATE DOMESTIC PRODUCT AT CURRENT PRICES; BASE YEAR 2011-12,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25,Unnamed: 26,Unnamed: 27,Unnamed: 28
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,As on 17.03.2025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,PER CAPITA NSDP AT CURRENT PRICES (In Rs),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,(% Growth over previous year),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,S. No.,State\UT,2011-12,2012-13,2013-14,2014-15,2015-16,2016-17,2017-18,2018-19,2019-20,2020-21,2021-22,2022-23,2023-24,2024-25,2012-13,2013-14,2014-15,2015-16,2016-17,2017-18,2018-19,2019-20,2020-21,2021-22,2022-23,2023-24,2024-25
4,(1),(2),(3),(4),(5),(6),(7),(8'),(9'),('10),('11),('12),('13),('14),('15),('16),('17),('18),('19),('20),('21),('22),('23),('24),('25),('26),('27),('28),('29)



Sheet: PC con.


,PER CAPITA NET STATE DOMESTIC PRODUCT AT CONSTANT PRICES; BASE YEAR 2011-12,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25,Unnamed: 26,Unnamed: 27,Unnamed: 28
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,As on 17.03.2025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,PER CAPITA NSDP AT CONSTANT PRICES (In Rs),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,(% Growth over previous year),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,S. No.,State\UT,2011-12,2012-13,2013-14,2014-15,2015-16,2016-17,2017-18,2018-19,2019-20,2020-21,2021-22,2022-23,2023-24,2024-25,2012-13,2013-14,2014-15,2015-16,2016-17,2017-18,2018-19,2019-20,2020-21,2021-22,2022-23,2023-24,2024-25
4,(1),(2),(3),(4),(5),(6),(7),(8'),(9),('10),('11),('12),('13),('14),('15),('16),('17),('18),('19),('20),('21),('22),('23),('24),('25),('26),('27),('28),('29)


In [31]:
# Load GSDP and per-capita NSDP sheets from the economy workbook
gsdp_df = pd.read_excel(
    "../data/raw/Statewise_SDP.xls",
    sheet_name="SDP-Curr.",
    header = 4
)

pc_df = pd.read_excel(
    "../data/raw/Statewise_SDP.xls",
    sheet_name="PC curr.",
    header = 4
)

In [32]:
gsdp_df.head(15)

,S. No.,State\UT,2011-12,2012-13,2013-14,2014-15,2015-16,2016-17,2017-18,2018-19,2019-20,2020-21,2021-22,2022-23,2023-24,2024-25,2012-13.1,2013-14.1,2014-15.1,2015-16.1,2016-17.1,2017-18.1,2018-19.1,2019-20.1,2020-21.1,2021-22.1,2022-23.1,2023-24.1,2024-25.1
0,(1),(2),(3),(4),(5),(6),(7),(8'),(9),('10),('11),('12),('13),('14),('15),('16),('17),('18),('19),('20),('21),('22),('23),('24),('25),('26),('27),('28),('29)
1,1,Andhra Pradesh,379402.03,411403.71,464272.01,524975.64,604228.619532,684415.87,786135.415564,873721.106301,925839.118118,978581.462125,1131628.952625,1309463.970318,1422093.896906,1593061.98,8.434768,12.850711,13.075014,15.096506,13.271012,14.862242,11.141298,5.965063,5.696707,15.639729,15.714958,8.601224,12.022278
2,2,Arunachal Pradesh,11062.69,12546.62,14581.07,17959.41,18509.16,19902.12,22474.75,25334.87,30023.65,30525.35,32705.26,35037.37,39040.7,NaN,13.413826,16.215124,23.169356,3.061069,7.525787,12.926412,12.725926,18.507219,1.671016,7.14131,7.130688,11.425886,NaN
3,3,Assam,143174.91,156864.24,177745.22,195723.15,227958.82,254382.36,283164.91,309336.32,346850.68,339802.98,410723.561023,479390.540997,570944.13,643666.69,9.561263,13.311498,10.114438,16.470034,11.591365,11.31468,9.242462,12.12737,-2.031912,20.871089,16.718539,19.097913,12.737246
4,4,Bihar,247143.960643,282367.930941,317101.34132,342950.940686,371601.790154,421051.49987,468746.310233,527975.819958,581855.484686,567814.024733,647394.348208,745439.667215,852621.202132,NaN,14.25241,12.300763,8.151842,8.354212,13.307177,11.327548,12.635728,10.204949,-2.413221,14.015209,15.144605,14.378298,NaN
5,5,Chhattisgarh,158073.820946,177511.329128,206833.18,221118.11,225162.99,262801.75,282737.44,327106.66,344672.04,352327.51,411613.35,458891.32,512107.49,567880.43,12.296475,16.518298,6.906498,1.829285,16.716229,7.585828,15.692729,5.369924,2.221088,16.826912,11.486015,11.596683,10.890866
6,6,Goa,42366.658073,38120.017077,35921.10428,47814.180193,55053.853142,62976.313594,69352.053703,71853.335311,75032.085208,74157.918705,81226.12592,93672.378766,106532.567346,NaN,-10.023545,-5.768394,33.108882,15.141268,14.390383,10.124029,3.606644,4.423942,-1.165057,9.531291,15.322967,13.728901,NaN
7,7,Gujarat,615606.074137,724495.361233,807623.193966,921773.146874,1029009.740813,1167155.579269,1329094.770869,1492155.713303,1617143.274963,1616106.361076,1920926.583759,2203418.967446,2425803.975032,NaN,17.688144,11.473894,14.134061,11.633729,13.425124,13.874688,12.268572,8.376308,-0.06412,18.861396,14.706048,10.092725,NaN
8,8,Haryana,297538.520682,347032.012669,399268.11619,437144.711348,495504.108617,561424.171155,638832.07542,698939.759266,738052.378164,730441.77335,877268.913603,974732.326528,1085510.28247,1213951.041963,16.634314,15.052243,9.486506,13.350132,13.303636,13.787775,9.408996,5.595993,-1.031174,20.101142,11.109867,11.364962,11.832293
9,9,Himachal Pradesh,72719.829321,82819.786161,94764.153845,103772.320292,114239.407089,125633.637518,138551.090377,148383.288252,159164.018861,151905.418701,170654.31081,191659.152551,210661.851559,232185.2323,13.888862,14.422118,9.50588,10.086588,9.973993,10.281843,7.096442,7.265461,-4.560453,12.342477,12.308416,9.91484,10.217028


### 4.1 Clean GSDP Data

Gross State Domestic Product is used as a proxy for economic scale. The latest available year is retained for the final investment opportunity dataset.


In [33]:
gsdp_clean = gsdp_df.iloc[:, :16].copy()

In [34]:
# Assign readable column names for yearly GSDP values
gsdp_clean.columns = [
    "S_No",
    "State",
    "GSDP_2011_12",
    "GSDP_2012_13",
    "GSDP_2013_14",
    "GSDP_2014_15",
    "GSDP_2015_16",
    "GSDP_2016_17",
    "GSDP_2017_18",
    "GSDP_2018_19",
    "GSDP_2019_20",
    "GSDP_2020_21",
    "GSDP_2021_22",
    "GSDP_2022_23",
    "GSDP_2023_24",
    "GSDP_2024_25"
]

In [35]:
gsdp_clean = gsdp_clean.dropna(subset=["State"])
gsdp_clean = gsdp_clean[gsdp_clean["State"] != "(2)"]
gsdp_clean = gsdp_clean.reset_index(drop=True)

In [36]:
# Standardize economy dataset state names for consistent merging
gsdp_clean["State"] = (
    gsdp_clean["State"]
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
    .str.title()
)

In [37]:
gsdp_clean = gsdp_clean[gsdp_clean["State"].isin(selected_states)].copy()
gsdp_clean = gsdp_clean.reset_index(drop=True)

In [38]:
gsdp_clean[["State", "GSDP_2023_24", "GSDP_2024_25"]]

,State,GSDP_2023_24,GSDP_2024_25
0,Andhra Pradesh,1422093.896906,1593061.98
1,Gujarat,2425803.975032,NaN
2,Haryana,1085510.28247,1213951.041963
3,Karnataka,2557241.345812,2883903.226981
4,Kerala,1139945.446934,NaN
5,Madhya Pradesh,1353808.97,1503395.4
6,Maharashtra,4055847.226926,4531518.415554
7,Punjab,745819.540606,809538.121437
8,Rajasthan,1521509.645283,1704338.546711
9,Tamil Nadu,2721570.512622,3103150.911687


In [39]:
# Keep the latest available GSDP indicator needed for the model
gsdp_final = gsdp_clean[["State", "GSDP_2023_24"]].copy()

In [40]:
gsdp_final

,State,GSDP_2023_24
0,Andhra Pradesh,1422093.896906
1,Gujarat,2425803.975032
2,Haryana,1085510.28247
3,Karnataka,2557241.345812
4,Kerala,1139945.446934
5,Madhya Pradesh,1353808.97
6,Maharashtra,4055847.226926
7,Punjab,745819.540606
8,Rajasthan,1521509.645283
9,Tamil Nadu,2721570.512622


In [41]:
pc_df.head(15)

,S. No.,State\UT,2011-12,2012-13,2013-14,2014-15,2015-16,2016-17,2017-18,2018-19,2019-20,2020-21,2021-22,2022-23,2023-24,2024-25,2012-13.1,2013-14.1,2014-15.1,2015-16.1,2016-17.1,2017-18.1,2018-19.1,2019-20.1,2020-21.1,2021-22.1,2022-23.1,2023-24.1,2024-25.1
0,(1),(2),(3),(4),(5),(6),(7),(8'),(9'),('10),('11),('12),('13),('14),('15),('16),('17),('18),('19),('20),('21),('22),('23),('24),('25),('26),('27),('28),('29)
1,1,Andhra Pradesh,68999.599025,74687.49878,82869.735747,93903.173854,108002.06954,120676.496352,138298.652958,154030.840128,160341.076666,168062.823341,193703.25977,219916.775056,237950.625141,266239.659807,8.243381,10.955297,13.314195,15.014291,11.735356,14.602808,11.375517,4.096736,4.815826,15.256459,13.532821,8.200307,11.888615
2,2,Arunachal Pradesh,73540.330697,82626.102418,94134.904996,114788.935282,116985.055096,124129.379686,138835.651587,155102.807487,182171.2773,181536.89384,190851.233766,199542.508039,220209.050351,NaN,12.354815,13.928773,21.940884,1.913181,6.10704,11.847535,11.716843,17.451953,-0.348235,5.130825,4.553952,10.356962,NaN
3,3,Assam,41141.859356,44599.177342,49733.890267,52894.577392,60816.531223,66329.598258,75151.472069,81033.702703,90122.708044,86946.750939,103371.080889,119192.432743,139783.009357,154221.635672,8.403407,11.513022,6.355198,14.976873,9.06508,13.300056,7.827166,11.216327,-3.524036,18.890102,15.305395,17.27507,10.329314
4,4,Bihar,21749.854247,24487.146102,26948.447653,28670.621362,30403.871576,34044.904862,36849.782064,40714.9169,44175.322015,42127.919073,47296.472315,53400.627642,60180.373079,NaN,12.585334,10.051402,6.390623,6.045388,11.975558,8.238758,10.488895,8.499109,-4.634721,12.268712,12.906154,12.696003,NaN
5,5,Chhattisgarh,55176.814451,60849.365954,69880.41241,72935.96274,72991.331393,83285.049941,88793.288472,102024.086402,106610.95067,106116.888828,123493.261038,134995.848454,148922.271231,162869.538472,10.28068,14.841644,4.372542,0.075914,14.102659,6.613718,14.900673,4.495864,-0.463425,16.374747,9.314344,10.316186,9.365468
6,6,Goa,259444.01415,234354.353065,215776.325117,289184.549785,334576.100608,378952.860121,411739.840885,423715.659578,435949.354098,423046.937012,459093.511961,532853.970501,585952.503727,NaN,-9.670549,-7.927324,34.020519,15.696396,13.263577,8.651995,2.908589,2.887242,-2.959614,8.520703,16.066543,9.964932,NaN
7,7,Gujarat,87480.619797,102826.236662,113138.646626,127016.591153,139253.970152,156294.971628,176961.391284,197456.867329,212427.841038,207324.440986,241584.382551,272450.708626,297722.298614,NaN,17.541733,10.028968,12.266317,9.634473,12.237354,13.222703,11.581891,7.581896,-2.402416,16.524796,12.776623,9.275656,NaN
8,8,Haryana,106084.695103,121268.815164,137769.649729,147382.113625,164962.654814,184982.003667,208437.274027,223021.552117,232530.490409,225137.197995,265163.469803,289212.649854,319363.013877,353181.554377,14.313205,13.606824,6.9772,11.928545,12.135685,12.679758,6.996963,4.263686,-3.179494,17.778613,9.069568,10.424981,10.589373
9,9,Himachal Pradesh,87720.987232,99730.363141,114094.740198,123299.421643,135512.121217,150289.941079,165496.889779,174803.939906,186559.204122,173564.849169,193391.858991,214488.590822,234782.257059,257212.047873,13.690425,14.403213,8.067577,9.904912,10.905165,10.118408,5.623701,6.724828,-6.965271,11.423402,10.9088,9.46142,9.553444


### 4.2 Clean Per-Capita NSDP Data

Per-capita NSDP is used as a proxy for income level and economic capacity. It complements GSDP by capturing market strength on a per-person basis.


In [42]:
pc_clean = pc_df.iloc[:, :16].copy()

In [43]:
# Assign readable column names for yearly per-capita NSDP values
pc_clean.columns = [
    "S_No",
    "State",
    "PC_NSDP_2011_12",
    "PC_NSDP_2012_13",
    "PC_NSDP_2013_14",
    "PC_NSDP_2014_15",
    "PC_NSDP_2015_16",
    "PC_NSDP_2016_17",
    "PC_NSDP_2017_18",
    "PC_NSDP_2018_19",
    "PC_NSDP_2019_20",
    "PC_NSDP_2020_21",
    "PC_NSDP_2021_22",
    "PC_NSDP_2022_23",
    "PC_NSDP_2023_24",
    "PC_NSDP_2024_25"
]


In [44]:
pc_clean = pc_clean.dropna(subset=["State"])
pc_clean = pc_clean[pc_clean["State"] != "(2)"]
pc_clean = pc_clean.reset_index(drop=True)

In [45]:
# Standardize per-capita NSDP state names for consistent merging
pc_clean["State"] = (
    pc_clean["State"]
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
    .str.title()
)

In [46]:
pc_clean = pc_clean[pc_clean["State"].isin(selected_states)].copy()
pc_clean = pc_clean.reset_index(drop=True)


In [47]:
pc_final = pc_clean[["State", "PC_NSDP_2023_24"]].copy()

In [48]:
pc_final

,State,PC_NSDP_2023_24
0,Andhra Pradesh,237950.625141
1,Gujarat,297722.298614
2,Haryana,319363.013877
3,Karnataka,339813.428685
4,Kerala,279750.717944
5,Madhya Pradesh,139712.786189
6,Maharashtra,278681.091346
7,Punjab,195030.509175
8,Rajasthan,166647.299904
9,Tamil Nadu,315219.594435


### 4.3 Merge Economic Indicators

The cleaned GSDP and per-capita NSDP tables are merged into one economy table before joining with the master dataset.


In [49]:
# Combine total economic scale and per-capita income indicators
economy_df = gsdp_final.merge(
    pc_final,
    on="State",
    how="inner"
)

economy_df

,State,GSDP_2023_24,PC_NSDP_2023_24
0,Andhra Pradesh,1422093.896906,237950.625141
1,Gujarat,2425803.975032,297722.298614
2,Haryana,1085510.28247,319363.013877
3,Karnataka,2557241.345812,339813.428685
4,Kerala,1139945.446934,279750.717944
5,Madhya Pradesh,1353808.97,139712.786189
6,Maharashtra,4055847.226926,278681.091346
7,Punjab,745819.540606,195030.509175
8,Rajasthan,1521509.645283,166647.299904
9,Tamil Nadu,2721570.512622,315219.594435


In [50]:
set(selected_states) - set(economy_df["State"])

set()

In [51]:
# Merge economy indicators into the charging infrastructure master table
master_df = master_df.merge(
    economy_df,
    on="State",
    how="left"
)

master_df.head()

,State,Charging_Station_Count,Total_Connectors,Fast_Charger_Count,Avg_Charger_Rating,District_Count,City_Count,Private_Charger_Rows,Govt_Charger_Rows,Connectors_per_Station,Fast_Charger_Share,GSDP_2023_24,PC_NSDP_2023_24
0,Andhra Pradesh,793,1089,340,23.229664,46,385,161,702,1.373266,42.875158,1422093.896906,237950.625141
1,Delhi,1977,3678,395,11.808317,24,3,2784,691,1.860395,19.979767,1112904.820708,459408.489922
2,Gujarat,1208,1792,574,26.449754,60,291,307,1116,1.483444,47.516556,2425803.975032,297722.298614
3,Haryana,936,3055,807,15.968648,39,173,2013,864,3.263889,86.217949,1085510.28247,319363.013877
4,Karnataka,6146,8512,761,8.630936,46,505,6288,1958,1.384966,12.382037,2557241.345812,339813.428685


In [52]:
master_df.to_csv("../data/cleaned/master_ev_charger_economy.csv", index=False)

## 5. Power Supply Data

Power supply indicators were explored as potential infrastructure readiness variables. The data is cleaned and merged here for experimentation, but these indicators should be used only if they meaningfully differentiate states in the final analysis.


In [53]:
# Load monthly power supply files for the selected states/UTs
power_files = glob.glob("../data/raw/power-supply-position-pe*.csv")

power_list = []

for file in power_files:
    temp_df = pd.read_csv(file)
    temp_df["Source_File"] = os.path.basename(file)
    power_list.append(temp_df)

power_df = pd.concat(power_list, ignore_index=True)

power_df.head()

,Category,Peak Demand,Peak Met,Source_File
0,Karnataka<br>Dec-2020,12123.109167,12102.000,power-supply-position-pe (1).csv
1,Karnataka<br>Jan-2021,12795.000000,12795.000,power-supply-position-pe (1).csv
2,Karnataka<br>Feb-2021,13440.577500,13432.000,power-supply-position-pe (1).csv
3,Karnataka<br>Mar-2021,14366.655000,14366.655,power-supply-position-pe (1).csv
4,Karnataka<br>Apr-2021,14158.000000,14158.000,power-supply-position-pe (1).csv


In [54]:
power_df.shape

(938, 4)

In [55]:
power_df.columns

Index(['Category', 'Peak Demand', 'Peak Met', 'Source_File'], dtype='object')

In [56]:
power_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 938 entries, 0 to 937
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Category     938 non-null    object 
 1   Peak Demand  938 non-null    float64
 2   Peak Met     938 non-null    float64
 3   Source_File  938 non-null    object 
dtypes: float64(2), object(2)
memory usage: 29.4+ KB


In [57]:
# Split combined category text into state and reporting period fields
power_df[["State", "Period"]] = power_df["Category"].str.split("<br>", expand=True)

In [58]:
power_df.tail(14)

,Category,Peak Demand,Peak Met,Source_File,State,Period
924,Madhya Pradesh<br>Nov-2024,17941.0,17923.0,power-supply-position-pe (9).csv,Madhya Pradesh,Nov-2024
925,Madhya Pradesh<br>Dec-2024,19371.0,19183.0,power-supply-position-pe (9).csv,Madhya Pradesh,Dec-2024
926,Madhya Pradesh<br>Jan-2025,18593.0,18580.0,power-supply-position-pe (9).csv,Madhya Pradesh,Jan-2025
927,Madhya Pradesh<br>Feb-2025,18596.0,18596.0,power-supply-position-pe (9).csv,Madhya Pradesh,Feb-2025
928,Madhya Pradesh<br>Mar-2025,16622.0,16622.0,power-supply-position-pe (9).csv,Madhya Pradesh,Mar-2025
929,Madhya Pradesh<br>Apr-2025,14527.0,14486.0,power-supply-position-pe (9).csv,Madhya Pradesh,Apr-2025
930,Madhya Pradesh<br>May-2025,13844.0,13844.0,power-supply-position-pe (9).csv,Madhya Pradesh,May-2025
931,Madhya Pradesh<br>Jun-2025,12733.0,12733.0,power-supply-position-pe (9).csv,Madhya Pradesh,Jun-2025
932,Madhya Pradesh<br>Jul-2025,12584.0,12584.0,power-supply-position-pe (9).csv,Madhya Pradesh,Jul-2025
933,Madhya Pradesh<br>Aug-2025,12869.0,12869.0,power-supply-position-pe (9).csv,Madhya Pradesh,Aug-2025


In [59]:
# Standardize power dataset state and period fields
power_df["State"] = (
    power_df["State"]
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
    .str.title()
)

power_df["Period"] = power_df["Period"].astype(str).str.strip()

In [60]:
power_df["Date"] = pd.to_datetime(
    power_df["Period"],
    format="%b-%Y",
    errors="coerce"
)

In [61]:
power_df = power_df.rename(columns={
    "Peak Demand": "Peak_Demand",
    "Peak Met": "Peak_Met"
})

In [62]:
power_df = power_df[power_df["State"].isin(selected_states)].copy()
power_df = power_df.reset_index(drop=True)

In [63]:
power_df["State"].unique()

array(['Karnataka', 'Haryana', 'Andhra Pradesh', 'Punjab', 'Maharashtra',
       'West Bengal', 'Uttar Pradesh', 'Delhi', 'Tamil Nadu', 'Gujarat',
       'Rajasthan', 'Kerala', 'Telangana', 'Madhya Pradesh'], dtype=object)

In [64]:
set(selected_states) - set(power_df["State"].unique())

set()

### 5.1 Latest Power Readiness Snapshot

For each state/UT, the latest available power supply observation is selected. Peak deficit is calculated to understand whether power availability could affect EV charging expansion.


In [65]:
# Keep the latest available power observation for each state/UT
power_latest = (
    power_df
    .sort_values("Date")
    .groupby("State", as_index=False)
    .tail(1)
    .reset_index(drop=True)
)

In [66]:
# Calculate peak power deficit and deficit percentage
power_latest["Peak_Deficit"] = (
    power_latest["Peak_Demand"] - power_latest["Peak_Met"]
)

power_latest["Peak_Deficit"] = power_latest["Peak_Deficit"].clip(lower=0)

power_latest["Peak_Deficit_Percentage"] = (
    power_latest["Peak_Deficit"] / power_latest["Peak_Demand"]
) * 100

power_latest["Power_Readiness_Score"] = (
    100 - power_latest["Peak_Deficit_Percentage"]
)

In [67]:
power_state = power_latest[
    [
        "State",
        "Period",
        "Date",
        "Peak_Demand",
        "Peak_Met",
        "Peak_Deficit",
        "Peak_Deficit_Percentage",
        "Power_Readiness_Score"
    ]
].copy()

In [68]:
power_state.sort_values("State")

,State,Period,Date,Peak_Demand,Peak_Met,Peak_Deficit,Peak_Deficit_Percentage,Power_Readiness_Score
9,Andhra Pradesh,Dec-2025,2025-12-01,11636.0,11586.0,50.0,0.429701,99.570299
5,Delhi,Dec-2025,2025-12-01,5505.0,5505.0,0.0,0.000000,100.000000
3,Gujarat,Dec-2025,2025-12-01,24369.0,24369.0,0.0,0.000000,100.000000
10,Haryana,Dec-2025,2025-12-01,9512.0,9512.0,0.0,0.000000,100.000000
11,Karnataka,Dec-2025,2025-12-01,17220.0,17220.0,0.0,0.000000,100.000000
1,Kerala,Dec-2025,2025-12-01,4567.0,4567.0,0.0,0.000000,100.000000
13,Madhya Pradesh,Dec-2025,2025-12-01,19851.0,19851.0,0.0,0.000000,100.000000
7,Maharashtra,Dec-2025,2025-12-01,28605.0,28605.0,0.0,0.000000,100.000000
8,Punjab,Dec-2025,2025-12-01,10172.0,10172.0,0.0,0.000000,100.000000
2,Rajasthan,Dec-2025,2025-12-01,19100.0,19100.0,0.0,0.000000,100.000000


In [69]:
# Merge power readiness indicators into the master table
master_df = master_df.merge(
    power_state,
    on="State",
    how="left"
)

master_df.head(12)

,State,Charging_Station_Count,Total_Connectors,Fast_Charger_Count,Avg_Charger_Rating,District_Count,City_Count,Private_Charger_Rows,Govt_Charger_Rows,Connectors_per_Station,Fast_Charger_Share,GSDP_2023_24,PC_NSDP_2023_24,Period,Date,Peak_Demand,Peak_Met,Peak_Deficit,Peak_Deficit_Percentage,Power_Readiness_Score
0,Andhra Pradesh,793,1089,340,23.229664,46,385,161,702,1.373266,42.875158,1422093.896906,237950.625141,Dec-2025,2025-12-01,11636.0,11586.0,50.0,0.429701,99.570299
1,Delhi,1977,3678,395,11.808317,24,3,2784,691,1.860395,19.979767,1112904.820708,459408.489922,Dec-2025,2025-12-01,5505.0,5505.0,0.0,0.000000,100.000000
2,Gujarat,1208,1792,574,26.449754,60,291,307,1116,1.483444,47.516556,2425803.975032,297722.298614,Dec-2025,2025-12-01,24369.0,24369.0,0.0,0.000000,100.000000
3,Haryana,936,3055,807,15.968648,39,173,2013,864,3.263889,86.217949,1085510.28247,319363.013877,Dec-2025,2025-12-01,9512.0,9512.0,0.0,0.000000,100.000000
4,Karnataka,6146,8512,761,8.630936,46,505,6288,1958,1.384966,12.382037,2557241.345812,339813.428685,Dec-2025,2025-12-01,17220.0,17220.0,0.0,0.000000,100.000000
5,Kerala,1394,2262,487,18.126961,26,294,601,1439,1.622669,34.935438,1139945.446934,279750.717944,Dec-2025,2025-12-01,4567.0,4567.0,0.0,0.000000,100.000000
6,Madhya Pradesh,1147,1432,335,17.510926,94,294,71,1128,1.248474,29.206626,1353808.97,139712.786189,Dec-2025,2025-12-01,19851.0,19851.0,0.0,0.000000,100.000000
7,Maharashtra,4183,6315,1240,15.794832,62,598,4109,1638,1.509682,29.643796,4055847.226926,278681.091346,Dec-2025,2025-12-01,28605.0,28605.0,0.0,0.000000,100.000000
8,Punjab,715,931,226,19.780584,46,223,88,700,1.302098,31.608392,745819.540606,195030.509175,Dec-2025,2025-12-01,10172.0,10172.0,0.0,0.000000,100.000000
9,Rajasthan,1531,1928,513,17.890078,99,383,280,1393,1.259308,33.507511,1521509.645283,166647.299904,Dec-2025,2025-12-01,19100.0,19100.0,0.0,0.000000,100.000000


In [70]:
master_df.isnull().sum()

State                      0
Charging_Station_Count     0
Total_Connectors           0
Fast_Charger_Count         0
Avg_Charger_Rating         0
District_Count             0
City_Count                 0
Private_Charger_Rows       0
Govt_Charger_Rows          0
Connectors_per_Station     0
Fast_Charger_Share         0
GSDP_2023_24               0
PC_NSDP_2023_24            0
Period                     0
Date                       0
Peak_Demand                0
Peak_Met                   0
Peak_Deficit               0
Peak_Deficit_Percentage    0
Power_Readiness_Score      0
dtype: int64

In [71]:
master_df.to_csv(
    "../data/cleaned/master_ev_charger_economy_power.csv",
    index=False
)

## 6. EV Registration Data

EV registration data is used to measure demand, market size, adoption intensity, and growth momentum. This is the core demand-side dataset for the investment opportunity model.


In [72]:
# Load year-wise EV registration and EV share data
ev_yoy_df = pd.read_excel("../data/raw/Electrical_Vehicle_YOY.xlsx")

In [73]:
ev_yoy_df.head()

,Year,State,Broad Category,EV vehicle registered,Share of EV in total vehicle registered (%)
0,2000-01,Andaman and Nicobar Islands,2 Wheeler,0.0,0.00
1,2000-01,Andhra Pradesh,2 Wheeler,0.0,0.00
2,2000-01,Arunachal Pradesh,2 Wheeler,0.0,0.00
3,2000-01,Assam,2 Wheeler,1.0,0.01
4,2000-01,Bihar,2 Wheeler,0.0,0.00


In [74]:
ev_yoy_df.tail()

,Year,State,Broad Category,EV vehicle registered,Share of EV in total vehicle registered (%)
11663,2026-27,West Bengal,Others,0.0,0.0
11664,NaN,NaN,NaN,NaN,NaN
11665,"Copyright © 2026, NITI Aayog",NaN,NaN,NaN,NaN
11666,https://iced.niti.gov.in,NaN,NaN,NaN,NaN
11667,The information on this platform is mainly tak...,NaN,NaN,NaN,NaN


In [75]:
ev_yoy_df[:11664]

,Year,State,Broad Category,EV vehicle registered,Share of EV in total vehicle registered (%)
0,2000-01,Andaman and Nicobar Islands,2 Wheeler,0.0,0.00
1,2000-01,Andhra Pradesh,2 Wheeler,0.0,0.00
2,2000-01,Arunachal Pradesh,2 Wheeler,0.0,0.00
3,2000-01,Assam,2 Wheeler,1.0,0.01
4,2000-01,Bihar,2 Wheeler,0.0,0.00
...,...,...,...,...,...
11659,2026-27,Telangana,Others,0.0,0.00
11660,2026-27,Tripura,Others,0.0,0.00
11661,2026-27,Uttar Pradesh,Others,0.0,0.00
11662,2026-27,Uttarakhand,Others,0.0,0.00


In [76]:
ev_yoy_df = ev_yoy_df[:11664]

In [77]:
ev_yoy_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11664 entries, 0 to 11663
Data columns (total 5 columns):
 #   Column                                       Non-Null Count  Dtype  
---  ------                                       --------------  -----  
 0   Year                                         11664 non-null  object 
 1   State                                        11664 non-null  object 
 2   Broad Category                               11664 non-null  object 
 3   EV vehicle registered                        11664 non-null  float64
 4   Share of EV in total vehicle registered (%)  11664 non-null  float64
dtypes: float64(2), object(3)
memory usage: 455.8+ KB


### 6.1 Clean EV Registration Fields

Column names, state names, and numeric fields are standardized so that yearly EV data can be aggregated consistently by state/UT.


In [78]:
# Clean source column names before renaming required fields
ev_yoy_df.columns = (
    ev_yoy_df.columns
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

ev_yoy_df.columns

Index(['Year', 'State', 'Broad Category', 'EV vehicle registered',
       'Share of EV in total vehicle registered (%)'],
      dtype='object')

In [79]:
# Rename EV columns to concise analysis names
ev_yoy_df = ev_yoy_df.rename(columns={
    "EV vehicle registered": "EV_Registrations",
    "Share of EV in total vehicle registered (%)": "EV_Share_Percentage",
    "Broad Category": "Broad_Category"
})

In [80]:
# Standardize EV dataset state names and handle known naming differences
ev_yoy_df["State"] = (
    ev_yoy_df["State"]
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
    .str.title()
)

state_mapping = {
    "Andaman And Nicobar Islands": "Andaman & Nicobar",
    "Nct Of Delhi": "Delhi",
    "National Capital Territory Of Delhi": "Delhi",
    "Delhi Nct": "Delhi",
    "Tamilnadu": "Tamil Nadu",
    "Uttar Pradesh ": "Uttar Pradesh",
    "West Bengal ": "West Bengal"
}

ev_yoy_df["State"] = ev_yoy_df["State"].replace(state_mapping)

In [81]:
# Convert EV registration and EV share fields to numeric values
ev_yoy_df["EV_Registrations"] = pd.to_numeric(
    ev_yoy_df["EV_Registrations"],
    errors="coerce"
)

ev_yoy_df["EV_Share_Percentage"] = pd.to_numeric(
    ev_yoy_df["EV_Share_Percentage"],
    errors="coerce"
)

In [82]:
# Extract starting year for chronological sorting and CAGR calculation
ev_yoy_df["Year_Start"] = (
    ev_yoy_df["Year"]
    .astype(str)
    .str[:4]
    .astype(int)
)

In [83]:
# Keep only the selected Version 1 states/UTs for EV analysis
ev_yoy_df = ev_yoy_df[ev_yoy_df["State"].isin(selected_states)].copy()
ev_yoy_df = ev_yoy_df.reset_index(drop=True)

In [84]:
ev_yoy_df["State"].nunique(), sorted(ev_yoy_df["State"].unique())

(14,
 ['Andhra Pradesh',
  'Delhi',
  'Gujarat',
  'Haryana',
  'Karnataka',
  'Kerala',
  'Madhya Pradesh',
  'Maharashtra',
  'Punjab',
  'Rajasthan',
  'Tamil Nadu',
  'Telangana',
  'Uttar Pradesh',
  'West Bengal'])

### 6.2 Build State-Year EV Metrics

The cleaned EV registration data is grouped by state and year to create annual EV registrations, EV share, and cumulative EV registrations.


In [85]:
ev_state_year = (
    ev_yoy_df
    .groupby(["State", "Year", "Year_Start"], as_index=False)
    .agg(
        Annual_EV_Registrations=("EV_Registrations", "sum"),
        Avg_EV_Share_Percentage=("EV_Share_Percentage", "mean")
    )
)

ev_state_year = ev_state_year.sort_values(["State", "Year_Start"])
ev_state_year.head()

,State,Year,Year_Start,Annual_EV_Registrations,Avg_EV_Share_Percentage
0,Andhra Pradesh,2000-01,2000,6.0,0.003333
1,Andhra Pradesh,2001-02,2001,4.0,0.001667
2,Andhra Pradesh,2002-03,2002,3.0,0.000833
3,Andhra Pradesh,2003-04,2003,4.0,0.001667
4,Andhra Pradesh,2004-05,2004,4.0,0.000833


In [86]:
ev_yoy_df = ev_yoy_df[ev_yoy_df["Year"] != "2026-27"].copy()

In [87]:
# Aggregate EV registrations at state-year level after excluding incomplete future year data
ev_state_year = (
    ev_yoy_df
    .groupby(["State", "Year", "Year_Start"], as_index=False)
    .agg(
        Annual_EV_Registrations=("EV_Registrations", "sum"),
        Avg_EV_Share_Percentage=("EV_Share_Percentage", "mean")
    )
)

ev_state_year = ev_state_year.sort_values(["State", "Year_Start"])

ev_state_year.head()

,State,Year,Year_Start,Annual_EV_Registrations,Avg_EV_Share_Percentage
0,Andhra Pradesh,2000-01,2000,6.0,0.003333
1,Andhra Pradesh,2001-02,2001,4.0,0.001667
2,Andhra Pradesh,2002-03,2002,3.0,0.000833
3,Andhra Pradesh,2003-04,2003,4.0,0.001667
4,Andhra Pradesh,2004-05,2004,4.0,0.000833


In [88]:
# Calculate cumulative EV registrations for each state/UT over time
ev_state_year["Cumulative_EV_Registrations"] = (
    ev_state_year
    .groupby("State")["Annual_EV_Registrations"]
    .cumsum()
)

ev_state_year.head()

,State,Year,Year_Start,Annual_EV_Registrations,Avg_EV_Share_Percentage,Cumulative_EV_Registrations
0,Andhra Pradesh,2000-01,2000,6.0,0.003333,6.0
1,Andhra Pradesh,2001-02,2001,4.0,0.001667,10.0
2,Andhra Pradesh,2002-03,2002,3.0,0.000833,13.0
3,Andhra Pradesh,2003-04,2003,4.0,0.001667,17.0
4,Andhra Pradesh,2004-05,2004,4.0,0.000833,21.0


In [89]:
ev_state_year["Cumulative_EV_Registrations"] = (
    ev_state_year
    .groupby("State")["Annual_EV_Registrations"]
    .cumsum()
)

ev_state_year.head()

,State,Year,Year_Start,Annual_EV_Registrations,Avg_EV_Share_Percentage,Cumulative_EV_Registrations
0,Andhra Pradesh,2000-01,2000,6.0,0.003333,6.0
1,Andhra Pradesh,2001-02,2001,4.0,0.001667,10.0
2,Andhra Pradesh,2002-03,2002,3.0,0.000833,13.0
3,Andhra Pradesh,2003-04,2003,4.0,0.001667,17.0
4,Andhra Pradesh,2004-05,2004,4.0,0.000833,21.0


### 6.3 Latest EV Snapshot and Growth Metrics

The latest complete year is used to create current EV demand indicators. CAGR is calculated from the base year to the latest year to capture long-term EV growth momentum.


In [90]:
# Extract the latest year snapshot for annual and cumulative EV metrics
latest_year = "2025-26"

ev_latest = ev_state_year[
    ev_state_year["Year"] == latest_year
].copy()

ev_latest = ev_latest.rename(columns={
    "Year": "Latest_EV_Year",
    "Annual_EV_Registrations": "Annual_EV_Registrations_2025_26",
    "Cumulative_EV_Registrations": "Cumulative_EV_Registrations_2025_26",
    "Avg_EV_Share_Percentage": "EV_Share_2025_26"
})

ev_latest = ev_latest[
    [
        "State",
        "Latest_EV_Year",
        "Annual_EV_Registrations_2025_26",
        "Cumulative_EV_Registrations_2025_26",
        "EV_Share_2025_26"
    ]
]

ev_latest.head()

,State,Latest_EV_Year,Annual_EV_Registrations_2025_26,Cumulative_EV_Registrations_2025_26,EV_Share_2025_26
25,Andhra Pradesh,2025-26,69495.0,218230.0,2.620833
51,Delhi,2025-26,107644.0,480158.0,17.646667
77,Gujarat,2025-26,86280.0,350476.0,2.314167
103,Haryana,2025-26,37137.0,170520.0,5.751667
129,Karnataka,2025-26,221441.0,762386.0,3.961667


In [91]:
# Calculate EV registration CAGR between base year and latest complete year
base_year = "2020-21"
final_year = "2025-26"

ev_base = ev_state_year[
    ev_state_year["Year"] == base_year
][["State", "Annual_EV_Registrations"]].rename(columns={
    "Annual_EV_Registrations": "Annual_EV_Registrations_2020_21"
})

ev_final = ev_state_year[
    ev_state_year["Year"] == final_year
][["State", "Annual_EV_Registrations"]].rename(columns={
    "Annual_EV_Registrations": "Annual_EV_Registrations_2025_26"
})

ev_growth = ev_base.merge(ev_final, on="State", how="inner")

years_diff = 5

ev_growth["EV_CAGR_2020_21_to_2025_26"] = np.where(
    ev_growth["Annual_EV_Registrations_2020_21"] > 0,
    (
        (ev_growth["Annual_EV_Registrations_2025_26"] /
         ev_growth["Annual_EV_Registrations_2020_21"]) ** (1 / years_diff) - 1
    ) * 100,
    np.nan
)

ev_growth["EV_CAGR_2020_21_to_2025_26"] = ev_growth["EV_CAGR_2020_21_to_2025_26"].round(2)

ev_growth.head()

,State,Annual_EV_Registrations_2020_21,Annual_EV_Registrations_2025_26,EV_CAGR_2020_21_to_2025_26
0,Andhra Pradesh,3134.0,69495.0,85.85
1,Delhi,11808.0,107644.0,55.58
2,Gujarat,1687.0,86280.0,119.66
3,Haryana,3029.0,37137.0,65.08
4,Karnataka,12986.0,221441.0,76.34


In [92]:
latest_year = "2025-26"

ev_latest = ev_state_year[
    ev_state_year["Year"] == latest_year
].copy()

ev_latest = ev_latest.rename(columns={
    "Year": "Latest_EV_Year",
    "Annual_EV_Registrations": "Annual_EV_Registrations_2025_26",
    "Cumulative_EV_Registrations": "Cumulative_EV_Registrations_2025_26",
    "Avg_EV_Share_Percentage": "EV_Share_2025_26"
})

ev_summary = ev_latest[
    [
        "State",
        "Latest_EV_Year",
        "Annual_EV_Registrations_2025_26",
        "Cumulative_EV_Registrations_2025_26",
        "EV_Share_2025_26"
    ]
]

ev_summary.head()

,State,Latest_EV_Year,Annual_EV_Registrations_2025_26,Cumulative_EV_Registrations_2025_26,EV_Share_2025_26
25,Andhra Pradesh,2025-26,69495.0,218230.0,2.620833
51,Delhi,2025-26,107644.0,480158.0,17.646667
77,Gujarat,2025-26,86280.0,350476.0,2.314167
103,Haryana,2025-26,37137.0,170520.0,5.751667
129,Karnataka,2025-26,221441.0,762386.0,3.961667


In [93]:
ev_summary.shape

(14, 5)

In [94]:
ev_summary.reset_index(drop= True)

,State,Latest_EV_Year,Annual_EV_Registrations_2025_26,Cumulative_EV_Registrations_2025_26,EV_Share_2025_26
0,Andhra Pradesh,2025-26,69495.0,218230.0,2.620833
1,Delhi,2025-26,107644.0,480158.0,17.646667
2,Gujarat,2025-26,86280.0,350476.0,2.314167
3,Haryana,2025-26,37137.0,170520.0,5.751667
4,Karnataka,2025-26,221441.0,762386.0,3.961667
5,Kerala,2025-26,115400.0,352288.0,4.072500
6,Madhya Pradesh,2025-26,158886.0,420139.0,6.601667
7,Maharashtra,2025-26,278993.0,980785.0,5.276667
8,Punjab,2025-26,46136.0,150500.0,7.508333
9,Rajasthan,2025-26,123153.0,480865.0,5.538333


In [95]:
ev_state_year.reset_index(drop= True)

,State,Year,Year_Start,Annual_EV_Registrations,Avg_EV_Share_Percentage,Cumulative_EV_Registrations
0,Andhra Pradesh,2000-01,2000,6.0,0.003333,6.0
1,Andhra Pradesh,2001-02,2001,4.0,0.001667,10.0
2,Andhra Pradesh,2002-03,2002,3.0,0.000833,13.0
3,Andhra Pradesh,2003-04,2003,4.0,0.001667,17.0
4,Andhra Pradesh,2004-05,2004,4.0,0.000833,21.0
...,...,...,...,...,...,...
359,West Bengal,2021-22,2021,6064.0,3.266667,48160.0
360,West Bengal,2022-23,2022,13260.0,4.166667,61420.0
361,West Bengal,2023-24,2023,26649.0,6.260000,88069.0
362,West Bengal,2024-25,2024,52330.0,7.112500,140399.0


In [96]:
# Save state-year EV trend data and latest EV summary data
ev_state_year.to_csv(
    "../data/cleaned/ev_state_year_annual_cumulative.csv",
    index=False
)

ev_summary.to_csv(
    "../data/cleaned/ev_summary_latest_growth.csv",
    index=False
)

## 7. Build Final Master Dataset

The final master dataset combines charging infrastructure, economic indicators, EV demand, cumulative EV market size, and EV growth metrics into one state-level table.


In [97]:
sorted(master_df["State"].unique())

['Andhra Pradesh',
 'Delhi',
 'Gujarat',
 'Haryana',
 'Karnataka',
 'Kerala',
 'Madhya Pradesh',
 'Maharashtra',
 'Punjab',
 'Rajasthan',
 'Tamil Nadu',
 'Telangana',
 'Uttar Pradesh',
 'West Bengal']

In [98]:
sorted(ev_summary["State"].unique())

['Andhra Pradesh',
 'Delhi',
 'Gujarat',
 'Haryana',
 'Karnataka',
 'Kerala',
 'Madhya Pradesh',
 'Maharashtra',
 'Punjab',
 'Rajasthan',
 'Tamil Nadu',
 'Telangana',
 'Uttar Pradesh',
 'West Bengal']

In [99]:
master_df.columns

Index(['State', 'Charging_Station_Count', 'Total_Connectors',
       'Fast_Charger_Count', 'Avg_Charger_Rating', 'District_Count',
       'City_Count', 'Private_Charger_Rows', 'Govt_Charger_Rows',
       'Connectors_per_Station', 'Fast_Charger_Share', 'GSDP_2023_24',
       'PC_NSDP_2023_24', 'Period', 'Date', 'Peak_Demand', 'Peak_Met',
       'Peak_Deficit', 'Peak_Deficit_Percentage', 'Power_Readiness_Score'],
      dtype='object')

In [100]:
# Merge latest EV demand metrics into the master dataset
master_df = master_df.merge(
    ev_summary,
    on="State",
    how="left"
)

In [101]:
# Merge EV growth metric into the master dataset
master_df = master_df.merge(
    ev_growth[["State", "EV_CAGR_2020_21_to_2025_26"]],
    on="State",
    how="left"
)

In [102]:
master_df.shape

(14, 25)

In [103]:
master_df.isnull().sum()

State                                  0
Charging_Station_Count                 0
Total_Connectors                       0
Fast_Charger_Count                     0
Avg_Charger_Rating                     0
District_Count                         0
City_Count                             0
Private_Charger_Rows                   0
Govt_Charger_Rows                      0
Connectors_per_Station                 0
Fast_Charger_Share                     0
GSDP_2023_24                           0
PC_NSDP_2023_24                        0
Period                                 0
Date                                   0
Peak_Demand                            0
Peak_Met                               0
Peak_Deficit                           0
Peak_Deficit_Percentage                0
Power_Readiness_Score                  0
Latest_EV_Year                         0
Annual_EV_Registrations_2025_26        0
Cumulative_EV_Registrations_2025_26    0
EV_Share_2025_26                       0
EV_CAGR_2020_21_

### 7.1 Create Charging Gap Metrics

Charging pressure metrics compare EV demand against available public charging infrastructure. Higher values indicate more EVs per station or connector, which can signal stronger infrastructure expansion need.


In [104]:
# Create cumulative EV charging pressure metrics
master_df["Cumulative_EVs_per_Charging_Station"] = (
    master_df["Cumulative_EV_Registrations_2025_26"] /
    master_df["Charging_Station_Count"]
)

master_df["Cumulative_EVs_per_Connector"] = (
    master_df["Cumulative_EV_Registrations_2025_26"] /
    master_df["Total_Connectors"]
)

In [105]:
# Create latest annual EV charging pressure metrics
master_df["Annual_EVs_2025_26_per_Charging_Station"] = (
    master_df["Annual_EV_Registrations_2025_26"] /
    master_df["Charging_Station_Count"]
)

master_df["Annual_EVs_2025_26_per_Connector"] = (
    master_df["Annual_EV_Registrations_2025_26"] /
    master_df["Total_Connectors"]
)

In [117]:
# Replace infinite values created by division with nulls for safer analysis
master_df = master_df.replace([np.inf, -np.inf], np.nan)

In [118]:
master_df.isnull().sum()

State                                      0
Latest_EV_Year                             0
Annual_EV_Registrations_2025_26            0
Cumulative_EV_Registrations_2025_26        0
EV_CAGR_2020_21_to_2025_26                 0
EV_Share_2025_26                           0
Charging_Station_Count                     0
Total_Connectors                           0
Fast_Charger_Count                         0
Avg_Charger_Rating                         0
District_Count                             0
City_Count                                 0
Private_Charger_Rows                       0
Govt_Charger_Rows                          0
Connectors_per_Station                     0
Fast_Charger_Share                         0
Cumulative_EVs_per_Charging_Station        0
Cumulative_EVs_per_Connector               0
Annual_EVs_2025_26_per_Charging_Station    0
Annual_EVs_2025_26_per_Connector           0
GSDP_2023_24                               0
PC_NSDP_2023_24                            0
Period    

### 7.2 Final Formatting and Column Order

The final table is rounded and reordered so it is easier to use in the EDA, scoring model, PCA, and regression notebooks.


In [119]:
# Round ratio and percentage metrics for cleaner reporting
round_columns = [
    "EV_CAGR_2020_21_to_2025_26",
    "EV_Share_2025_26",
    "Cumulative_EVs_per_Charging_Station",
    "Cumulative_EVs_per_Connector",
    "Annual_EVs_2025_26_per_Charging_Station",
    "Annual_EVs_2025_26_per_Connector",
    "Connectors_per_Station",
    "Fast_Charger_Share",
    "Peak_Deficit_Percentage",
    "Power_Readiness_Score"
]

for col in round_columns:
    if col in master_df.columns:
        master_df[col] = master_df[col].round(2)

In [120]:
# Arrange final master dataset columns in a logical analysis order
final_column_order = [
    "State",

    "Latest_EV_Year",
    "Annual_EV_Registrations_2025_26",
    "Cumulative_EV_Registrations_2025_26",
    "EV_CAGR_2020_21_to_2025_26",
    "EV_Share_2025_26",

    "Charging_Station_Count",
    "Total_Connectors",
    "Fast_Charger_Count",
    "Avg_Charger_Rating",
    "District_Count",
    "City_Count",
    "Private_Charger_Rows",
    "Govt_Charger_Rows",
    "Connectors_per_Station",
    "Fast_Charger_Share",

    "Cumulative_EVs_per_Charging_Station",
    "Cumulative_EVs_per_Connector",
    "Annual_EVs_2025_26_per_Charging_Station",
    "Annual_EVs_2025_26_per_Connector",

    "GSDP_2023_24",
    "PC_NSDP_2023_24",

    "Period",
    "Date",
    "Peak_Demand",
    "Peak_Met",
    "Peak_Deficit",
    "Peak_Deficit_Percentage",
    "Power_Readiness_Score"
]

master_df = master_df[
    [col for col in final_column_order if col in master_df.columns]
]

## 8. Final Validation

Before saving the dataset, basic validation checks are performed to confirm the number of rows, column types, missing values, and state/UT coverage.


In [121]:
master_df.head()

,State,Latest_EV_Year,Annual_EV_Registrations_2025_26,Cumulative_EV_Registrations_2025_26,EV_CAGR_2020_21_to_2025_26,EV_Share_2025_26,Charging_Station_Count,Total_Connectors,Fast_Charger_Count,Avg_Charger_Rating,District_Count,City_Count,Private_Charger_Rows,Govt_Charger_Rows,Connectors_per_Station,Fast_Charger_Share,Cumulative_EVs_per_Charging_Station,Cumulative_EVs_per_Connector,Annual_EVs_2025_26_per_Charging_Station,Annual_EVs_2025_26_per_Connector,GSDP_2023_24,PC_NSDP_2023_24,Period,Date,Peak_Demand,Peak_Met,Peak_Deficit,Peak_Deficit_Percentage,Power_Readiness_Score
0,Andhra Pradesh,2025-26,69495.0,218230.0,85.85,2.62,793,1089,340,23.229664,46,385,161,702,1.37,42.88,275.20,200.39,87.64,63.82,1.422094e+06,237950.625141,Dec-2025,2025-12-01,11636.0,11586.0,50.0,0.43,99.57
1,Delhi,2025-26,107644.0,480158.0,55.58,17.65,1977,3678,395,11.808317,24,3,2784,691,1.86,19.98,242.87,130.55,54.45,29.27,1.112905e+06,459408.489922,Dec-2025,2025-12-01,5505.0,5505.0,0.0,0.00,100.00
2,Gujarat,2025-26,86280.0,350476.0,119.66,2.31,1208,1792,574,26.449754,60,291,307,1116,1.48,47.52,290.13,195.58,71.42,48.15,2.425804e+06,297722.298614,Dec-2025,2025-12-01,24369.0,24369.0,0.0,0.00,100.00
3,Haryana,2025-26,37137.0,170520.0,65.08,5.75,936,3055,807,15.968648,39,173,2013,864,3.26,86.22,182.18,55.82,39.68,12.16,1.085510e+06,319363.013877,Dec-2025,2025-12-01,9512.0,9512.0,0.0,0.00,100.00
4,Karnataka,2025-26,221441.0,762386.0,76.34,3.96,6146,8512,761,8.630936,46,505,6288,1958,1.38,12.38,124.05,89.57,36.03,26.02,2.557241e+06,339813.428685,Dec-2025,2025-12-01,17220.0,17220.0,0.0,0.00,100.00


In [122]:
master_df.shape

(14, 29)

In [123]:
master_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 29 columns):
 #   Column                                   Non-Null Count  Dtype         
---  ------                                   --------------  -----         
 0   State                                    14 non-null     object        
 1   Latest_EV_Year                           14 non-null     object        
 2   Annual_EV_Registrations_2025_26          14 non-null     float64       
 3   Cumulative_EV_Registrations_2025_26      14 non-null     float64       
 4   EV_CAGR_2020_21_to_2025_26               14 non-null     float64       
 5   EV_Share_2025_26                         14 non-null     float64       
 6   Charging_Station_Count                   14 non-null     int64         
 7   Total_Connectors                         14 non-null     int64         
 8   Fast_Charger_Count                       14 non-null     int64         
 9   Avg_Charger_Rating                       14 n

In [124]:
master_df.isnull().sum()

State                                      0
Latest_EV_Year                             0
Annual_EV_Registrations_2025_26            0
Cumulative_EV_Registrations_2025_26        0
EV_CAGR_2020_21_to_2025_26                 0
EV_Share_2025_26                           0
Charging_Station_Count                     0
Total_Connectors                           0
Fast_Charger_Count                         0
Avg_Charger_Rating                         0
District_Count                             0
City_Count                                 0
Private_Charger_Rows                       0
Govt_Charger_Rows                          0
Connectors_per_Station                     0
Fast_Charger_Share                         0
Cumulative_EVs_per_Charging_Station        0
Cumulative_EVs_per_Connector               0
Annual_EVs_2025_26_per_Charging_Station    0
Annual_EVs_2025_26_per_Connector           0
GSDP_2023_24                               0
PC_NSDP_2023_24                            0
Period    

In [125]:
master_df["State"].nunique()

14

In [126]:
master_df["State"].value_counts()

State
Andhra Pradesh    1
Delhi             1
Gujarat           1
Haryana           1
Karnataka         1
Kerala            1
Madhya Pradesh    1
Maharashtra       1
Punjab            1
Rajasthan         1
Tamil Nadu        1
Telangana         1
Uttar Pradesh     1
West Bengal       1
Name: count, dtype: int64

## 9. Save Final Cleaned Dataset

The final cleaned dataset is saved as the main input for the next notebook. This file should be the single source of truth for EDA, investment scoring, PCA, and exploratory regression.


In [127]:
# Save final cleaned master dataset used for EDA, scoring, PCA, and regression
master_df.to_csv(
    "../data/cleaned/master_ev_investment_dataset_updated.csv",
    index=False
)

## Cleaning Notebook Summary

This notebook produced the final state-level dataset for the EV charging investment opportunity analysis. The dataset combines demand-side EV adoption metrics, supply-side public charging infrastructure metrics, charging pressure indicators, and economic variables.

The cleaned master dataset is now ready for exploratory analysis, opportunity scoring, PCA, and exploratory regression.
